# Uji R32 — Run all tiga notebook pada Colab bersih

Pilih **Runtime → Run all**. Unggah `R32_ANALISIS_PORTABEL.zip` saat diminta. Runtime CPU cukup; tidak memakai Drive. File keluaran berisi notebook yang benar-benar dieksekusi dan laporan runtime. Setelah selesai, ambil screenshot tabel ringkasan untuk bukti pembimbing. Jangan menyatakan lulus sebelum seluruh sel selesai.

In [ ]:
import sys, os, json, hashlib, zipfile, subprocess, tempfile
from pathlib import Path
import google.colab
from google.colab import files
assert os.environ.get('COLAB_RELEASE_TAG') or Path('/opt/google/colab').exists(), 'Jalankan pada runtime Google Colab asli'
uploaded = files.upload()
archive = next((Path(name) for name in uploaded if name.endswith('.zip')), None)
assert archive is not None, 'Unggah ZIP analisis portabel'
del uploaded
assert hashlib.sha256(archive.read_bytes()).hexdigest() == 'e70ef8033f71aeca2894431f338f554b06b65e1a96215b7c0d2d1515c17159d1', 'SHA256 ZIP tidak cocok'

In [ ]:
directory = Path(tempfile.mkdtemp(prefix='joy_r32_'))
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
    for name in z.namelist():
        assert (directory/name).resolve().is_relative_to(directory), 'Path ZIP tidak aman'
    z.extractall(directory)
ROOT = directory/'joy_revisi_ii'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT/'requirements.txt')], check=True)
print('Paket diekstrak dan dependensi dipasang. Tidak ada training.')

In [ ]:
output = Path('/content/JOY_R32_HASIL_COLAB')
assert not output.exists(), 'Gunakan runtime Colab baru atau simpan hasil sebelumnya lebih dahulu'
subprocess.run([sys.executable, str(ROOT/'tools/run_clean_notebooks.py'), '--root', str(ROOT), '--output', str(output)], check=True)
result = json.loads((output/'RUNTIME_VERIFICATION.json').read_text())
assert result['status'] == 'verified' and result['colab_verified'] is True
from IPython.display import display, HTML
rows = ''.join('<tr><td>'+r['notebook']+'</td><td>'+str(r['errors'])+'</td><td>'+str(r['seconds'])+'</td></tr>' for r in result['notebooks'])
display(HTML('<h2>R32 — Run all Colab lulus</h2><p>'+result['created_at']+'</p><table><tr><th>Notebook</th><th>Error</th><th>Detik</th></tr>'+rows+'</table><p>Ambil screenshot sel ini dan simpan ZIP bukti berikut.</p>'))

In [ ]:
import shutil
archive_result = shutil.make_archive('/content/JOY_R32_BUKTI_COLAB', 'zip', output)
files.download(archive_result)